In [76]:
import pandas as pd

# --- Step 1: Load the data ---
data = pd.read_csv("CAR_data_V2.csv")

/var/folders/h7/c_5v7d7s0w7d_6fry6l74vyc0000gn/T/ipykernel_79433/1560013642.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("CAR_data_V2.csv")


## Create data for number of pickups

In [77]:
# --- Step 2: Ensure datetime format ---
data['Activity Start Timestamp'] = pd.to_datetime(data['Activity Start Timestamp'], errors='coerce')

In [78]:
# Step 1: Create a flag for each Contact Session ID indicating if an agent name ever appears
agent_present = (
    data.groupby('Contact Session ID')['Agent Name']
    .apply(lambda x: x.notna().any())
    .rename('Agent_Present')
)

# Step 2: Merge that flag back into the main dataframe
data = data.merge(agent_present, on='Contact Session ID', how='left')

# Step 3: Apply the logic, using both per-row and session-level info
data['Picked_Up'] = data.apply(
    lambda row: (
        # Rule 1: If Activity Name is 'CallbackRetry' and the session never had an agent → NOT picked up
        False if row['Activity Name'] == 'CallbackRetry' and not row['Agent_Present']
        # Rule 2: If the session had an agent or row indicates screen pop → picked up
        else True if row['Agent_Present'] or row['Activity Name'] == 'LegalServerScreenPop'
        # Rule 3: Otherwise, mark as not picked up
        else False
    ),
    axis=1
)

In [79]:
# --- Step 4: Save for Power BI ---
data.to_csv("CAR_data_pickups_v1.csv", index=False)

## Create New Rows For Hierarchical Filtering 

In [ ]:
# --- Step 1: Load the data ---
df = pd.read_csv("CAR_data_V2.csv")

In [ ]:
# --- Step 2: Ensure datetime format ---
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')

# --- Step 3: Group by Contact Session ID and aggregate ---
grouped = (
    df.groupby('Contact Session ID')
    .agg(
        Call_Start_Time=('Activity Start Timestamp', 'min'),   # earliest timestamp per call
        Starting_Hour=('hour', 'min'),                         # earliest hour per call
        Count=('Activity Start Timestamp', lambda x: len(set(x))),  # number of unique activity timestamps
        Call_Duration=('Activity Start Timestamp', 
                       lambda x: (max(x) - min(x)).total_seconds() / 60 if len(x) > 1 else 0)
    )
    .reset_index()
)

In [ ]:
# --- Step 4: Create Year, Month, Day Columns ---
grouped['Year'] = grouped['Call_Start_Time'].dt.year 
grouped['Month'] = grouped['Call_Start_Time'].dt.month_name()
grouped['Day'] = grouped['Call_Start_Time'].dt.day_name()

In [ ]:
# --- Step 5: Save for Power BI ---
grouped.to_csv("CAR_data_transformed_v2.csv", index=False)